# post CE catalog from Kruckow et al. 2021

Notebook to download their catalog and pour the data in our format.

https://ui.adsabs.harvard.edu/abs/2021ApJ...920...86K/abstract

**Data is downloaded to /data/from_other 

================================================================================
Title: A Catalog of Potential Post Common Envelope Binaries 
Authors: Kruckow M.U., Neunteufel P.G., Di Stefano R., Gao Y., Kobayashi C. 
================================================================================
Description of contents: Three ascii tables asssociated with Table 3. The first
 is the catalog, datafile3a.txt. It contains 848 entries in the post
 common envelope binary catalog. The second file, datafile3b.txt, has the same
 format as the primary catalog but has the 12 sources that were removed from 
 the catalog. These systems which should not be used by the readers as they
 failed the criteria to become part of the catalog. They are provided for 
 completeness. The final file, refs.txt contains the bibcodes for all the 
 citation codes given in the other files.

System requirements: None. 

Additional comments: Note that the primary catalog has 839 sources. A few 
 systems have several reported values which are inconsistent. These systems 
 have two entries, where the second one is marked by a "#" at the beginning 
 of the line and in the flag column. See last paragraph of section 2.4.2 for
 more details.


 **Note that the term primary refers to the donor of the most recent common
 envelope phase.**


In [35]:
import pandas as pd
import json
import numpy as np
import h5py
from pathlib import Path
import sys
import re

proj_root = Path('/Users/liekevanson/Documents/Projects/post_mt_review').resolve()
if str(proj_root) not in sys.path:
    sys.path.insert(0, str(proj_root))

from paths import RESULT_TABLES, RAW_JSON_DIR, DATA_DIR


In [36]:
# Define column specifications based on the ReadMe file
# Format: (start_byte-1, end_byte) since Python uses 0-indexed positions
colspecs = [
    (0, 29),      # Name
    (30, 46),     # P (Period)
    (47, 57),     # M1 (Primary mass)
    (58, 65),     # M2 (Secondary mass)
    (66, 69),     # Type1
    (70, 78),     # Type2
    (79, 89),     # b_M1
    (90, 97),     # B_M1
    (98, 105),    # b_M2
    (106, 114),   # B_M2
    (115, 123),   # q
    (124, 132),   # b_q
    (133, 141),   # B_q
    (142, 151),   # a
    (152, 161),   # b_a
    (162, 171),   # B_a
    (172, 184),   # e
    (185, 196),   # b_e
    (197, 208),   # B_e     # Note there is a typo in the datafile3a.txt stating both b_e and B_e are Lower boundary in e
    (209, 215),   # i
    (216, 222),   # b_i
    (223, 229),   # B_i
    (230, 238),   # R1
    (239, 247),   # b_R1
    (248, 256),   # B_R1
    (257, 266),   # R2
    (267, 276),   # b_R2
    (277, 286),   # B_R2
    (287, 293),   # T1
    (294, 300),   # b_T1
    (301, 310),   # B_T1
    (311, 319),   # T2
    (320, 328),   # b_T2
    (329, 337),   # B_T2
    (338, 349),   # L1
    (350, 360),   # b_L1
    (361, 372),   # B_L1
    (373, 384),   # L2
    (385, 394),   # b_L2
    (395, 404),   # B_L2
    (405, 411),   # log(g1)
    (412, 418),   # b_log(g1)
    (419, 425),   # B_log(g1)
    (426, 433),   # log(g2)
    (434, 441),   # b_log(g2)
    (442, 449),   # B_log(g2)
    (450, 459),   # Age
    (460, 469),   # b_Age
    (470, 479),   # B_Age
    (480, 487),   # Flag
    (488, 505),   # S-RAdeg (Simbad RA)
    (506, 523),   # S-DEdeg (Simbad Dec)
    (524, 543),   # Gaia
    (544, 562),   # G-RAdeg (Gaia RA)
    (563, 582),   # G-DEdeg (Gaia Dec)
    (583, 592),   # D (Distance)
    (593, 600),   # b_D
    (601, 609),   # B_D
    (610, 647),   # Cite
    (648, 773),   # Ref
    (774, 875),   # Note
]

# Column names
names = [
    'Name', 'P', 'M1', 'M2', 'Type1', 'Type2',
    'b_M1', 'B_M1', 'b_M2', 'B_M2',
    'q', 'b_q', 'B_q',
    'a', 'b_a', 'B_a',
    'e', 'b_e', 'B_e',
    'i', 'b_i', 'B_i',
    'R1', 'b_R1', 'B_R1',
    'R2', 'b_R2', 'B_R2',
    'T1', 'b_T1', 'B_T1',
    'T2', 'b_T2', 'B_T2',
    'L1', 'b_L1', 'B_L1',
    'L2', 'b_L2', 'B_L2',
    'log_g1', 'b_log_g1', 'B_log_g1',
    'log_g2', 'b_log_g2', 'B_log_g2',
    'Age', 'b_Age', 'B_Age',
    'Flag', 'S_RAdeg', 'S_DEdeg', 'Gaia', 'G_RAdeg', 'G_DEdeg',
    'D', 'b_D', 'B_D',
    'Cite', 'Ref', 'Note'
]

In [37]:
# Read the catalogs
table_a = DATA_DIR / "from_others" / "apjac13act3_mrt" / "datafile3a.txt"
# table_b = DATA_DIR / "from_others" / "apjac13act3_mrt" / "datafile3b.txt" # Should not be used (includes removed sources)


# Read datafile3a (primary catalog - 848 entries)
# Skip the header lines (first 94 lines contain the description and column specs)
Kruckow_a = pd.read_fwf(table_a, colspecs=colspecs, names=names, skiprows=94, na_values=[''])

# Remove rows that start with '#' (duplicate entries that should be marked) -- see readme.txt
Kruckow_a['is_duplicate'] = Kruckow_a['Name'].str.startswith('#', na=False)
print(f"Found {Kruckow_a['is_duplicate'].sum()} duplicate/alternate entries marked with '#'")

print(f"\nDatafile 3a shape: {Kruckow_a.shape}")
Kruckow_a.head()

Found 9 duplicate/alternate entries marked with '#'

Datafile 3a shape: (847, 62)


,Name,P,M1,M2,Type1,Type2,b_M1,B_M1,b_M2,B_M2,...,Gaia,G_RAdeg,G_DEdeg,D,b_D,B_D,Cite,Ref,Note,is_duplicate
0,ZTFJ1539+5027,0.004801,0.610,0.210,WD,WD,0.588,0.627,0.195,0.224,...,1.402815e+18,234.883971,50.460756,2.1026,0.9540,NaN,bcf+19,Burdge et al. (2019),NaN,False
1,ZTFJ2243+5242,0.006110,0.349,0.384,WD,WD,0.275,0.442,0.310,0.498,...,2.002025e+18,340.929043,52.701660,0.8195,0.5325,1.7781,bcf+20,Burdge et al. (2020b),SEDfit,False
2,V407Vul/RXJ1914.4+2456,0.006590,0.800,0.177,WD,NaN,0.700,0.900,0.106,0.248,...,2.023676e+18,288.608706,24.945351,8.7108,2.8313,NaN,"rhc02,kks+18",Ramsay et al. (2002); Kupfer et al. (2018),AM CVn type,False
3,ESCet,0.007178,0.800,0.161,WD,NaN,0.700,0.900,0.097,0.225,...,2.462596e+18,30.217729,-9.408804,1.7313,1.5497,1.9612,"epw+05,kks+18",Espaillat et al. (2005); Kupfer et al. (2018),AM CVn type,False
4,4U1820-303,0.007930,0.055,1.580,WD,NS,NaN,NaN,1.520,1.640,...,4.046461e+18,275.918886,-30.361116,0.3763,0.3174,0.4620,"spw87,gwc+10",Stella et al. (1987); Guver et al. (2010),globular cluster; UCXB,False


In [38]:
display(Kruckow_a)

,Name,P,M1,M2,Type1,Type2,b_M1,B_M1,b_M2,B_M2,...,Gaia,G_RAdeg,G_DEdeg,D,b_D,B_D,Cite,Ref,Note,is_duplicate
0,ZTFJ1539+5027,0.004801,0.610,0.210,WD,WD,0.588,0.627,0.195,0.224,...,1.402815e+18,234.883971,50.460756,2.1026,0.9540,NaN,bcf+19,Burdge et al. (2019),NaN,False
1,ZTFJ2243+5242,0.006110,0.349,0.384,WD,WD,0.275,0.442,0.310,0.498,...,2.002025e+18,340.929043,52.701660,0.8195,0.5325,1.7781,bcf+20,Burdge et al. (2020b),SEDfit,False
2,V407Vul/RXJ1914.4+2456,0.006590,0.800,0.177,WD,NaN,0.700,0.900,0.106,0.248,...,2.023676e+18,288.608706,24.945351,8.7108,2.8313,NaN,"rhc02,kks+18",Ramsay et al. (2002); Kupfer et al. (2018),AM CVn type,False
3,ESCet,0.007178,0.800,0.161,WD,NaN,0.700,0.900,0.097,0.225,...,2.462596e+18,30.217729,-9.408804,1.7313,1.5497,1.9612,"epw+05,kks+18",Espaillat et al. (2005); Kupfer et al. (2018),AM CVn type,False
4,4U1820-303,0.007930,0.055,1.580,WD,NS,NaN,NaN,1.520,1.640,...,4.046461e+18,275.918886,-30.361116,0.3763,0.3174,0.4620,"spw87,gwc+10",Stella et al. (1987); Guver et al. (2010),globular cluster; UCXB,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
842,S1040/NGC2682SAND1040,42.800000,0.220,1.500,WD,G4,NaN,NaN,NaN,NaN,...,6.049177e+17,132.849010,11.830359,0.8894,0.8563,0.9251,lab+97,Landsman et al. (1997),formed via stable MT?; uncertain parallax,False
843,AYCet,56.800000,0.550,2.090,WD,G,NaN,NaN,NaN,NaN,...,2.531780e+18,19.150753,-2.500652,0.0747,0.0742,0.0753,sfg85,Simon et al. (1985),formed via stable MT?,False
844,PSRJ1713+0747,67.825138,0.286,1.310,WD,NS,0.274,0.298,1.200,1.420,...,4.395035e+18,258.457413,7.792077,18.8679,3.4722,NaN,zsd+15,Zhu et al. (2015),formed via stable MT?,False
845,KOI-3278,88.180520,0.634,1.042,WD,G,0.579,0.681,0.984,1.070,...,2.052729e+18,291.524936,38.455910,0.7249,0.7155,0.7345,ka14,Kruse & Agol (2014),formed via stable MT?,False


In [39]:
print(np.unique(Kruckow_a['Type1']))
# print(np.unique(Kruckow_a['Type2']))

type2_counts = Kruckow_a['Type2'].str.strip().value_counts(dropna=True)
print(type2_counts)

['WD' 'sdB' 'sdO']
Type2
WD          202
M           107
WD/MS        46
NS           45
A/F          38
M4           30
BD           29
MS           24
M5           21
M3           20
A            14
K5           10
K             8
M2            7
G             5
M6            5
K0            5
K2            4
A2            4
K3            4
M0            4
K4            3
WD/NS/BH      3
M7            3
NS/BH         3
A3            3
F             2
M8            2
M1            2
F0            1
A0            1
A9            1
G0            1
A6            1
A1            1
WD/M          1
G6            1
B             1
G2            1
G7            1
A4            1
F8            1
sdB           1
BH            1
G8            1
K7            1
K8            1
BD/MS         1
G9            1
G4            1
Name: count, dtype: int64


In [40]:
# Load reference code → ADS bibcode mapping
refs_path = DATA_DIR / "from_others" / "apjac13act3_mrt" / "refs.txt"
refs_map = {}
with open(refs_path) as fh:
    for line in fh:
        m = re.search(r"=\s*([^\s]+)\s*\[(.+?)\]", line)
        if m:
            code = m.group(1).strip().rstrip(';')
            bib = m.group(2).strip()
            refs_map[code] = bib
print(f"Loaded {len(refs_map)} reference codes from {refs_path}")

Loaded 467 reference codes from /Users/liekevanson/Documents/Projects/post_mt_review/data/from_others/apjac13act3_mrt/refs.txt


In [46]:
def kruckow_to_catalog_schema(row):
    """
    Transform a Kruckow catalog row to our JSON schema format.

    IMPORTANT: Kruckow uses _1 for donor (primary from CE),
    but our schema uses _2 for the presumed donor.
    So we swap: Kruckow _1 → our _2, Kruckow _2 → our _1
    """

    # Helper function to create triplet [err-, value, err+]
    def make_triplet(lower_bound, value, upper_bound):
        if pd.isna(value):
            return None
        err_minus = lower_bound if not pd.isna(lower_bound) else None
        err_plus = upper_bound if not pd.isna(upper_bound) else None
        return [err_minus, value, err_plus]

    # Helper to split names by '/' - returns primary name and list of alternates
    def parse_names(name_str):
        if pd.isna(name_str):
            return "", []
        # Remove leading '#' if present
        name_str = name_str.strip().lstrip('#')
        names = [n.strip() for n in name_str.split('/')]
        primary = names[0] if names else ""
        alternates = names[1:] if len(names) > 1 else []
        return primary, alternates

    # Helper to map Kruckow types to evol_type
    def map_evol_type(type_str):
        if pd.isna(type_str):
            return None
        t = type_str.strip()
        mapping = {
            'WD': 'WD',
            'NS': 'NS',
            'BH': 'BH',
            'sdB': 'He-star',
            'sdO': 'He-star',
            'MS': 'MS',
            'BD': None,  # Brown dwarf not in our schema
            'BS': None,  # Blue straggler - not an evolutionary state
        }
        if t in mapping:
            return mapping[t]
        # If it looks like a spectral type (OBAFGKMLTY... with digits), map to MS
        if re.match(r'^[OBAFGKMLTY][0-9]', t, flags=re.IGNORECASE):
            return 'MS'
        return None

    # Helper to get obs_type (for spectral types or observational classes)
    def get_obs_type(type_str):
        if pd.isna(type_str):
            return None
        t = type_str.strip()
        # Return as-is for observational classification
        obs_classes = ['WD', 'sdB', 'sdO', 'MS', 'NS', 'BH', 'BD', 'BS']
        if t in obs_classes:
            return t
        # Otherwise assume it's a spectral type
        return t

    entry = {}

    # System Name - extract primary and alternate names
    primary_name, alternate_names = parse_names(row['Name'])
    entry['System Name'] = primary_name

    # Coordinates - prefer Gaia over SIMBAD
    ra_val = row['G_RAdeg'] if not pd.isna(row['G_RAdeg']) else row['S_RAdeg']
    dec_val = row['G_DEdeg'] if not pd.isna(row['G_DEdeg']) else row['S_DEdeg']
    entry['RA'] = [None, ra_val, None] if not pd.isna(ra_val) else None
    entry['Dec'] = [None, dec_val, None] if not pd.isna(dec_val) else None

    # Period (convert from days as stored)
    entry['Period'] = make_triplet(None, row['P'], None)  # No uncertainties given

    # Eccentricity
    # Note there is a typo in the datafile3a.txt stating both b_e and B_e are Lower boundary in e
    entry['Eccentricity'] = make_triplet(row['b_e'], row['e'], row['B_e'])

    # SWAP: Kruckow _1 (donor) → our _2, Kruckow _2 → our _1
    entry['M1'] = make_triplet(row['b_M2'], row['M2'], row['B_M2'])  # Their M2 → our M1
    entry['M2'] = make_triplet( row['b_M1'], row['M1'], row['B_M1'])  # Their M1 → our M2

    entry['Mass Function'] = None

    # Evolutionary types (swapped)
    entry['evol_type_1'] = map_evol_type(row['Type2'])  # Their Type2 → our evol_type_1
    entry['evol_type_2'] = map_evol_type(row['Type1'])  # Their Type1 → our evol_type_2

    # Observational types (swapped)
    entry['obs_type_1'] = get_obs_type(row['Type2'])  # Their Type2 → our obs_type_1
    entry['obs_type_2'] = get_obs_type(row['Type1'])  # Their Type1 → our obs_type_2

    # System class: hot subdwarf if donor (their Type1) is sdB/sdO, else generic post-CE
    donor_type = str(row['Type1']).strip() if not pd.isna(row['Type1']) else None
    if donor_type in ['sdB', 'sdO']:
        entry['system_class'] = 'Hot subdwarf'
    else:
        if donor_type == 'WD':
            entry['system_class'] = "WD + MS"
        else:
            entry['system_class'] = 'post-CE binary'

    # Detection method - infer from available data
    detection_methods = []
    if not pd.isna(row['M1']) and not pd.isna(row['M2']):
        detection_methods.append('RV')
    if not pd.isna(row['i']):
        detection_methods.append('EB')
    if not pd.isna(row['G_RAdeg']):
        detection_methods.append('Astrometry')
    entry['Detection Method'] = detection_methods if detection_methods else None

    # Reference - parse bibcodes from Cite field using refs_map; always include catalog paper
    refs = ['2021ApJ...920...86K'] # Kruckow et al. 2021 catalog paper that we draw from
    if not pd.isna(row['Cite']):
        for code in [c.strip() for c in row['Cite'].split(',') if c.strip()]:
            bib = refs_map.get(code)
            refs.append(bib if bib else code)

    # Deduplicate while preserving order
    seen = set()
    deduped_refs = []
    for r in refs:
        if r not in seen:
            seen.add(r)
            deduped_refs.append(r)
    entry['Reference'] = deduped_refs

    # Notes - combine alternate names, Flag, Ref, and Note fields
    notes = []
    if alternate_names:
        notes.append(f"Alternate names: {', '.join(alternate_names)}")
    if not pd.isna(row['Flag']):
        flag_str = row['Flag'].strip()
        if flag_str and flag_str != '#':
            notes.append(f"Flags: {flag_str}")
    if not pd.isna(row['Note']):
        note_str = row['Note'].strip()
        if note_str:
            notes.append(note_str)
    entry['Notes'] = '; '.join(notes) if notes else None

    # Simbad URL
    if not pd.isna(ra_val) and not pd.isna(dec_val):
        entry['Simbad'] = f"http://simbad.u-strasbg.fr/simbad/sim-coo?Coord={ra_val}+{dec_val}&Radius=10"
    else:
        entry['Simbad'] = None

    return entry

In [47]:
# Transform datafile3a to catalog format
# Exclude rows marked with '#' as duplicates (only keep primary entry)
df_primary = Kruckow_a[~Kruckow_a['is_duplicate']].copy()

print(f"Converting {len(df_primary)} systems from Kruckow catalog...")
catalog_entries = []

WDWD_WDNS_counter = 0
for idx, row in df_primary.iterrows():
    entry = kruckow_to_catalog_schema(row)

    # We care specifically about post "Mass transfer 1" systems
    # Exclude systems that have presumably undergone a 2nd mt phase

    # Exclude WDWD and WDNS systems 
    WDNS_permutations_bool = np.logical_and(entry['evol_type_1'] in ['WD','NS'], entry['evol_type_2'] in ['WD', 'NS'])

    # Also exclude all sdOB + WD/NS/BH systems
    sdOB_compobj_bool = np.logical_and(entry['obs_type_2'] in ['sdO','sdB'], entry['evol_type_1'] in ['WD', 'NS', 'BH'])
    
    if np.logical_or(WDNS_permutations_bool,sdOB_compobj_bool):
        WDWD_WDNS_counter +=1
        continue
    catalog_entries.append(entry)

print(f"Skipped {WDWD_WDNS_counter} WDWD and WDNS systems")
print(f"Successfully converted {len(catalog_entries)} systems")
print(f"\nSample entry:")
catalog_entries[0]

Converting 838 systems from Kruckow catalog...
Skipped 244 WDWD and WDNS systems
Successfully converted 594 systems

Sample entry:


{'System Name': 'V407Vul',
 'RA': [None, 288.608706444482, None],
 'Dec': [None, 24.9453507748492, None],
 'Period': [None, 0.00659022, None],
 'Eccentricity': None,
 'M1': [0.106, 0.177, 0.248],
 'M2': [0.7, 0.8, 0.9],
 'Mass Function': None,
 'evol_type_1': None,
 'evol_type_2': 'WD',
 'obs_type_1': None,
 'obs_type_2': 'WD',
 'system_class': 'WD + MS',
 'Detection Method': ['RV', 'EB', 'Astrometry'],
 'Reference': ['2021ApJ...920...86K',
  '2002MNRAS.332L...7R',
  '2018MNRAS.480..302K'],
 'Notes': 'Alternate names: RXJ1914.4+2456; Flags: MT/S; AM CVn type',
 'Simbad': 'http://simbad.u-strasbg.fr/simbad/sim-coo?Coord=288.608706444482+24.9453507748492&Radius=10'}

In [48]:
# Save to JSON file
output_file = RAW_JSON_DIR / "Kruckow2021_postCE.raw.json"

with open(output_file, "w") as f:
    f.write("[\n")
    for i, system in enumerate(catalog_entries):
        line = json.dumps(system, separators=(",", ": "), ensure_ascii=False)
        f.write("  " + line)
        if i < len(catalog_entries) - 1:
            f.write(",\n")
        else:
            f.write("\n")
    f.write("]\n")


print(f"Saved {len(catalog_entries)} entries to {output_file}")

Saved 594 entries to /Users/liekevanson/Documents/Projects/post_mt_review/data/result_tables/raw_json/Kruckow2021_postCE.raw.json


In [49]:
# Get unique combinations of evol_type_1 and evol_type_2
unique_evol_combos = set((entry['evol_type_1'], entry['evol_type_2']) for entry in catalog_entries)

tot_count = 0   
print(f"Found {len(unique_evol_combos)} unique evolutionary type combinations:\n")
for combo in sorted(unique_evol_combos, key=lambda x: (x[0] or '', x[1] or '')):
    count = sum(1 for e in catalog_entries if (e['evol_type_1'], e['evol_type_2']) == combo)
    evol1 = combo[0] if combo[0] else 'None'
    evol2 = combo[1] if combo[1] else 'None'
    print(f"  {evol1:10s} + {evol2:10s} : {count:4d} systems")
    tot_count += count

print(f"Totals to : {tot_count}  ")


Found 5 unique evolutionary type combinations:

  None       + He-star    :   94 systems
  None       + WD         :  333 systems
  He-star    + He-star    :    1 systems
  MS         + He-star    :   13 systems
  MS         + WD         :  153 systems
Totals to : 594  


In [50]:
# Get unique combinations of obs_type_1 and obs_type_2, grouping spectral types
def group_obs_type(obs_type):
    """Group spectral types together, keep compact objects separate"""
    if obs_type in ['NS', 'WD', 'sdB', 'sdO', 'BH', 'BD', 'BS', 'MS', None]:
        return obs_type
    else:
        return 'Spectral type'

# Create grouped combinations
grouped_combos = {}
for entry in catalog_entries:
    grouped_type1 = group_obs_type(entry['obs_type_1'])
    grouped_type2 = group_obs_type(entry['obs_type_2'])
    key = (grouped_type1, grouped_type2)
    grouped_combos[key] = grouped_combos.get(key, 0) + 1

tot_count = 0
print(f"Found {len(grouped_combos)} unique observational type combinations (with spectral types grouped):\n")
for combo in sorted(grouped_combos.keys(), key=lambda x: (x[0] or '', x[1] or '')):
    count = grouped_combos[combo]
    obs1 = combo[0] if combo[0] else 'None'
    obs2 = combo[1] if combo[1] else 'None'
    print(f"  {obs1:15s} + {obs2:15s} : {count:4d} systems")
    tot_count += count

print(f"Totals to : {tot_count}  ")

Found 10 unique observational type combinations (with spectral types grouped):

  None            + WD              :  170 systems
  None            + sdB             :    2 systems
  BD              + WD              :   24 systems
  BD              + sdB             :    5 systems
  MS              + WD              :   15 systems
  MS              + sdB             :    9 systems
  Spectral type   + WD              :  277 systems
  Spectral type   + sdB             :   84 systems
  Spectral type   + sdO             :    7 systems
  sdB             + sdB             :    1 systems
Totals to : 594  
